# 🎬 Short Drama Studio — make pictures move (free GPU)
1. Top menu **Runtime → Change runtime type → T4 GPU → Save**.
2. Tap ▶ on each step, one after another.
3. Each picture becomes a ~4 second moving clip. Download it, then in the app tap **Add moving clip** on that scene.

In [ ]:
#@title Step 1: Set up (tap ▶; if it says "restarting", wait 10 s and tap ▶ again)
import os
if not os.path.exists('/content/.sds_ready'):
    !pip -q install -U diffusers transformers accelerate sentencepiece imageio imageio-ffmpeg
    open('/content/.sds_ready', 'w').write('1')
    print('🔄 Installed. Restarting the session — wait 10 seconds, then tap ▶ on Step 1 again.')
    os.kill(os.getpid(), 9)
import torch, gc
assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > T4 GPU, then run again.'
from diffusers import LTXImageToVideoPipeline
from diffusers.utils import export_to_video
pipe = LTXImageToVideoPipeline.from_pretrained('Lightricks/LTX-Video', torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()
pipe.vae.enable_tiling()
print('✅ Ready — now tap ▶ on Step 2')


In [ ]:
#@title Step 2: Upload your scene pictures
from google.colab import files
up = files.upload()
print('Got', len(up), 'picture(s)')

In [ ]:
#@title Step 3: Make them move
what_happens = "the character moves naturally and talks with expressive face and body language" #@param {type:"string"}
seconds = 4 #@param [3, 4, 5] {type:"raw"}
style = "Pixar style 3D animation, cinematic, smooth motion, high quality"
neg = "worst quality, blurry, distorted, jittery, deformed face, extra limbs, text, watermark"
from PIL import Image, ImageOps
from google.colab import files
for name in up:
    img = ImageOps.fit(Image.open(name).convert('RGB'), (480, 832))
    n = int(seconds * 24) // 8 * 8 + 1
    print('Animating', name, '… (a few minutes each)')
    frames = pipe(image=img, prompt=f'{what_happens}. {style}', negative_prompt=neg, width=480, height=832, num_frames=n, num_inference_steps=30).frames[0]
    out = name.rsplit('.', 1)[0] + '_moving.mp4'
    export_to_video(frames, out, fps=24)
    files.download(out)
    gc.collect(); torch.cuda.empty_cache()
print('✅ Done')